# Samsung PRISM 26VI05 - Phi-3.5 Vision comparison on Colab
1. Runtime -> Change runtime type -> **GPU**. Run the smoke test to check T4 memory use before long jobs; a CPU runtime cannot train this pipeline.
2. Runtime -> Run all. The model branch is cloned into `MyDrive/prism_ladder_phi35`; images, checkpoints and reports
   are saved there too, so after a disconnect just Run all again and every stage resumes where it stopped.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os, subprocess
workdir = '/content/drive/MyDrive/prism_ladder_phi35'
if not os.path.exists(workdir):
    subprocess.run(['git', 'clone', '--depth', '1', '--branch', 'codex/phi35-model', 'https://github.com/Sanjana432/Samsung-prism-san.git', workdir], check=True)
os.chdir(workdir)
subprocess.run(['git', 'pull', '--ff-only'], check=True)
!git log --oneline -1
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
!pip install -q -r requirements.txt

In [ ]:
# Phi native multi-image processor: 4 crops per image, eager attention on T4.
import torch, yaml
assert torch.cuda.is_available(), 'Choose a GPU runtime before running model stages.'
mem = torch.cuda.get_device_properties(0).total_memory / 2**30
with open('configs/config.yaml') as f:
    cfg = yaml.safe_load(f)
assert cfg['model']['base'] == 'microsoft/Phi-3.5-vision-instruct'
if mem < 20:
    cfg['inference']['batch_size'] = 2
    with open('configs/config.yaml', 'w') as f:
        yaml.safe_dump(cfg, f, sort_keys=False)
print(f'{mem:.0f} GB -> crops', cfg['model']['num_crops'], 'batch', cfg['inference']['batch_size'])

In [ ]:
import json
if not os.path.exists('data/metadata/ladder.jsonl'):
    !python scripts/01_generate_dataset.py
!python scripts/02_build_splits.py
if not os.path.exists('pilot/images'):
    !python scripts/make_pilot_showcase.py
!python scripts/00_selftest.py

In [ ]:
!python scripts/gpu_smoke_test.py

In [ ]:
!python scripts/03_prompt_study_and_baseline.py

In [ ]:
!python scripts/04_train_qlora.py

In [ ]:
!python scripts/05_infer.py --adapter outputs/best_adapter --name finetuned_qlora --split test
!python scripts/06_evaluate.py --models baseline_zero_shot finetuned_qlora --split test

In [ ]:
from IPython.display import Markdown, Image, display
display(Markdown(open('reports/evaluation_report.md').read()))
display(Image('reports/figures/mae_per_metric.png'))